# Model Tuning

Tune promising model families with a reproducible validation strategy.

# Logistic Regression baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scores = []
auc_scores = []
balanced_scores = []

for train_idx, val_idx in skf.split(X_imputed, y_class):

    X_train = X_imputed.iloc[train_idx]
    X_val = X_imputed.iloc[val_idx]

    y_train = y_class.iloc[train_idx]
    y_val = y_class.iloc[val_idx]

    logistic_model.fit(X_train, y_train)

    pred = logistic_model.predict(X_val)
    prob = logistic_model.predict_proba(X_val)[:, 1]

    # Probability of Invalid
    invalid_prob = (
        logistic_model.predict_proba(X_val)[:, 
        list(logistic_model.classes_).index("Invalid")]
    )

    f1_scores.append(
        f1_score(
            y_val,
            pred,
            pos_label="Invalid"
        )
    )

    auc_scores.append(
        roc_auc_score(
            (y_val == "Invalid").astype(int),
            invalid_prob
        )
    )

    balanced_scores.append(
        balanced_accuracy_score(y_val, pred)
    )

print("Logistic Regression")
print("-------------------")
print("F1:", np.mean(f1_scores))
print("ROC-AUC:", np.mean(auc_scores))
print("Balanced Accuracy:", np.mean(balanced_scores))

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scores = []
auc_scores = []
balanced_scores = []

for train_idx, val_idx in skf.split(X_imputed, y_class):

    X_train = X_imputed.iloc[train_idx]
    X_val = X_imputed.iloc[val_idx]

    y_train = y_class.iloc[train_idx]
    y_val = y_class.iloc[val_idx]

    logistic_model.fit(X_train, y_train)

    pred = logistic_model.predict(X_val)
    prob = logistic_model.predict_proba(X_val)[:, 1]

    # Probability of Invalid
    invalid_prob = (
        logistic_model.predict_proba(X_val)[:, 
        list(logistic_model.classes_).index("Invalid")]
    )

    f1_scores.append(
        f1_score(
            y_val,
            pred,
            pos_label="Invalid"
        )
    )

    auc_scores.append(
        roc_auc_score(
            (y_val == "Invalid").astype(int),
            invalid_prob
        )
    )

    balanced_scores.append(
        balanced_accuracy_score(y_val, pred)
    )

print("Logistic Regression")
print("-------------------")
print("F1:", np.mean(f1_scores))
print("ROC-AUC:", np.mean(auc_scores))
print("Balanced Accuracy:", np.mean(balanced_scores))

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [ ]:
f1_scores = []
auc_scores = []
balanced_scores = []

for train_idx, val_idx in skf.split(X_imputed, y_class):

    X_train = X_imputed.iloc[train_idx]
    X_val = X_imputed.iloc[val_idx]

    y_train = y_class.iloc[train_idx]
    y_val = y_class.iloc[val_idx]

    rf_classifier.fit(X_train, y_train)

    pred = rf_classifier.predict(X_val)

    invalid_index = list(
        rf_classifier.classes_
    ).index("Invalid")

    invalid_prob = rf_classifier.predict_proba(
        X_val
    )[:, invalid_index]

    f1_scores.append(
        f1_score(
            y_val,
            pred,
            pos_label="Invalid"
        )
    )

    auc_scores.append(
        roc_auc_score(
            (y_val == "Invalid").astype(int),
            invalid_prob
        )
    )

    balanced_scores.append(
        balanced_accuracy_score(y_val, pred)
    )

print("Random Forest")
print("-------------")
print("F1:", np.mean(f1_scores))
print("ROC-AUC:", np.mean(auc_scores))
print("Balanced Accuracy:", np.mean(balanced_scores))

**The Valid/Invalid boundary is highly nonlinear and interaction-dependent.**

***Feature Importance***

In [ ]:
rf_classifier.fit(
    X_imputed,
    y_class
)

importance = pd.Series(
    rf_classifier.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

display(
    importance.head(20)
)

plt.figure(figsize=(10, 7))

importance.head(15).sort_values().plot(
    kind="barh"
)

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

It suggests that absolute sensor values aren't the primary signal for validity.

In [ ]:
X_reg = train[feature_cols].copy()
X_test_reg = test[feature_cols].copy()

X_reg = pd.DataFrame(
    imputer.fit_transform(X_reg),
    columns=X_reg.columns
)

X_test_reg = pd.DataFrame(
    imputer.transform(X_test_reg),
    columns=X_test_reg.columns
)

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, val_idx in kf.split(X_reg):

    X_train = X_reg.iloc[train_idx]
    X_val = X_reg.iloc[val_idx]

    y_train = y_reg.iloc[train_idx]
    y_val = y_reg.iloc[val_idx]

    linear_model.fit(X_train, y_train)

    pred = linear_model.predict(X_val)

    mae_scores.append(
        mean_absolute_error(y_val, pred)
    )

    rmse_scores.append(
        np.sqrt(mean_squared_error(y_val, pred))
    )

    r2_scores.append(
        r2_score(y_val, pred)
    )

print("Linear Regression")
print("-----------------")
print("MAE:", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R²:", np.mean(r2_scores))

*R^2 is negative, it means the linear model is performing worse than simply predicting the mean on the validation folds.*

# Stronger Regression model - Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_regressor = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

mae_scores = []
rmse_scores = []
r2_scores = []

for train_idx, val_idx in kf.split(X_reg):

    X_train = X_reg.iloc[train_idx]
    X_val = X_reg.iloc[val_idx]

    y_train = y_reg.iloc[train_idx]
    y_val = y_reg.iloc[val_idx]

    rf_regressor.fit(X_train, y_train)

    pred = rf_regressor.predict(X_val)

    mae_scores.append(
        mean_absolute_error(y_val, pred)
    )

    rmse_scores.append(
        np.sqrt(mean_squared_error(y_val, pred))
    )

    r2_scores.append(
        r2_score(y_val, pred)
    )

print("Random Forest Regression")
print("------------------------")
print("MAE:", np.mean(mae_scores))
print("RMSE:", np.mean(rmse_scores))
print("R²:", np.mean(r2_scores))

***Analysis***

| Task      | Model               |       MAE |      RMSE |        R² |        F1 |   ROC-AUC | Balanced Acc. |
| --------- | ------------------- | --------: | --------: | --------: | --------: | --------: | ------------: |
| Validity  | Logistic Regression |         — |         — |         — | **0.267** |     0.565 |         0.577 |
| Validity  | **Random Forest**   |         — |         — |         — | **0.965** | **0.985** |     **0.973** |
| Reference | Linear Regression   | **5.304** |    29.347 |    -15.58 |         — |         — |             — |
| Reference | **Random Forest**   | **0.992** | **2.185** | **0.954** |         — |         — |             — |

**Conclusion** : Random Forest is substantially better suited to this dataset than linear models.

**Why:**

- Sensor relationships dominate validity prediction.
- The relationship between operating conditions, sensors, and `Reference_Parameter` is nonlinear.
- Random Forest captures nonlinearities and feature interactions without extensive preprocessing.
- With approximately 1,000 training samples, a tree ensemble is a sensible choice over a neural network.

# Additional Test for Leakage check 

- **Goal:** Verify whether engineered residual features caused data leakage during cross-validation.
- Residuals may have been calculated using the entire dataset before splitting.
- Compare:
    - **Current approach:** Residuals calculated globally, then cross-validation performed.
    - **Leakage-safe approach:** Residual relationships calculated within each training fold and applied to validation data.
- Interpretation:
    - Similar performance indicates the high F1 score is genuine.
    - A substantial performance drop indicates the previous CV score was optimistic.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score

base_features = [
    "Applied_Voltage_kV",
    "Load_Current_A",
    "Ambient_Temperature_C",
    "Test_Duration_min",
    "Sensor_S1",
    "Sensor_S2",
    "Sensor_S3",
    "Sensor_S4"
]

X_raw = train[base_features].copy()
y = train["Validity_Label"].copy()

In [ ]:
residual_cols = [
    "S2_S1_residual",
    "S3_S1_residual",
    "S3_S2_residual"
]

print(train[residual_cols].corr())

***Leakage-safe CV test***

In [ ]:
from sklearn.base import clone

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

f1_scores = []
auc_scores = []
balanced_scores = []

for train_idx, val_idx in skf.split(X_raw, y):

    fold_train = train.iloc[train_idx].copy()
    fold_val = train.iloc[val_idx].copy()


    from sklearn.linear_model import LinearRegression

    residual_pairs = [
        ("Sensor_S1", "Sensor_S2", "S2_S1_residual"),
        ("Sensor_S1", "Sensor_S3", "S3_S1_residual"),
        ("Sensor_S2", "Sensor_S3", "S3_S2_residual")
    ]

    for x_col, y_col, residual_col in residual_pairs:

        mask = fold_train[[x_col, y_col]].notna().all(axis=1)

        relationship = LinearRegression()

        relationship.fit(
            fold_train.loc[mask, [x_col]],
            fold_train.loc[mask, y_col]
        )

        # Train residual
        train_mask = fold_train[x_col].notna()

        fold_train.loc[train_mask, residual_col] = (
            fold_train.loc[train_mask, y_col]
            - relationship.predict(
                fold_train.loc[train_mask, [x_col]]
            )
        )

        # Validation residual
        val_mask = fold_val[x_col].notna()

        fold_val.loc[val_mask, residual_col] = (
            fold_val.loc[val_mask, y_col]
            - relationship.predict(
                fold_val.loc[val_mask, [x_col]]
            )
        )


    engineered = [
        "S2_S1_residual",
        "S3_S1_residual",
        "S3_S2_residual"
    ]

    features = base_features + engineered

    X_train = fold_train[features]
    X_val = fold_val[features]

    y_train = fold_train["Validity_Label"]
    y_val = fold_val["Validity_Label"]

    # Impute using TRAIN fold only
    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(X_train)
    X_val = imputer.transform(X_val)

    # --------------------------------
    # Model
    # --------------------------------

    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    invalid_idx = list(model.classes_).index("Invalid")

    prob = model.predict_proba(X_val)[:, invalid_idx]

    f1_scores.append(
        f1_score(y_val, pred, pos_label="Invalid")
    )

    auc_scores.append(
        roc_auc_score(
            (y_val == "Invalid").astype(int),
            prob
        )
    )

    balanced_scores.append(
        balanced_accuracy_score(y_val, pred)
    )

print("Leakage-safe CV")
print("----------------")
print("F1:", np.mean(f1_scores))
print("ROC-AUC:", np.mean(auc_scores))
print("Balanced Accuracy:", np.mean(balanced_scores))

# Takeaways

The classification task exhibits nonlinear relationships between sensor measurements and validity. Relationship-derived features capturing deviations between sensor pairs substantially improved predictive performance. A Random Forest classifier was selected because it can capture nonlinear interactions and threshold-based sensor anomalies while remaining appropriate for the relatively small dataset. Leakage-safe cross-validation achieved an F1 score of 0.916, ROC-AUC of 0.982, and balanced accuracy of 0.931.

***RF hyperparameter tuning***

In [ ]:
print("y_class:")
print(type(y_class))
print(y_class.shape)
print(y_class.head() if hasattr(y_class, "head") else y_class[:10])

print("\n" + "="*50)

print("y_reg:")
print(type(y_reg))
print(y_reg.shape)
print(y_reg.head() if hasattr(y_reg, "head") else y_reg[:10])

print("\n" + "="*50)

print("y_train:")
print(type(y_train))
print(y_train.shape)
print(y_train[:10])

print("\n" + "="*50)

print("y_val:")
print(type(y_val))
print(y_val.shape)
print(y_val[:10])

In [ ]:
# ============================================================
# RF CLASSIFICATION
# HYPERPARAMETER TUNING + LEAKAGE-SAFE THRESHOLD OPTIMIZATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score


# Use the training data already defined earlier in this notebook.
X_rf = X_imputed.copy()
y_rf = y_class.copy()

y_rf_binary = (y_rf == "Invalid").astype(int)

print("TRAINING TARGET DISTRIBUTION")
print("----------------------------")
print(
    pd.Series(y_rf_binary)
    .map({0: "Valid", 1: "Invalid"})
    .value_counts()
)

In [ ]:
# ------------------------------------------------------------
# Median imputation
# ------------------------------------------------------------

imputer_rf = SimpleImputer(strategy="median")
X_rf_imputed = imputer_rf.fit_transform(X_rf)

print("Missing values BEFORE imputation")
print("---------------------------------")
print("Training :", X_rf.isna().sum().sum())

print("\nMissing values AFTER imputation")
print("--------------------------------")
print("Training :", np.isnan(X_rf_imputed).sum())

In [ ]:
# ------------------------------------------------------------
# Stratified 5-fold Cross Validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold stratified cross-validation configured.")

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING
# ============================================================

rf = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [500, 1000],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.5]
}

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    error_score="raise"
)

print("Starting Random Forest hyperparameter tuning...\n")

grid.fit(X_rf_imputed, y_rf_binary)

best_rf = grid.best_estimator_

print("\nBEST RANDOM FOREST")
print("==================")
print("\nBest parameters:")
print(grid.best_params_)
print(f"\nBest CV Invalid F1: {grid.best_score_:.4f}")
print("\nModel classes:")
print(best_rf.classes_)

**Used Cloud TPU**

Starting Random Forest hyperparameter tuning...

Fitting 5 folds for each of 48 candidates, totalling 240 fits

BEST RANDOM FOREST
==================

Best parameters:
{'max_depth': 20, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}

Best CV Invalid F1: 0.9686

Model classes:
[0 1]

In [ ]:
# ============================================================
# LEAKAGE-SAFE THRESHOLD OPTIMIZATION
# ============================================================

# Generate out-of-fold probabilities
# This avoids leakage because every prediction is generated on data that was not used to fit that fold's model.
oof_probabilities = cross_val_predict(
    best_rf,
    X_rf_imputed,
    y_rf_binary,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)

# Column 1 = Invalid probability
invalid_probability_oof = oof_probabilities[:, 1]

print("OOF INVALID PROBABILITY")
print("=======================")
print(f"Minimum : {invalid_probability_oof.min():.4f}")
print(f"Maximum : {invalid_probability_oof.max():.4f}")
print(f"Mean    : {invalid_probability_oof.mean():.4f}")

thresholds = np.linspace(0.05, 0.95, 181)
best_threshold = 0.5
best_f1 = -np.inf

for threshold in thresholds:
    pred = (invalid_probability_oof >= threshold).astype(int)
    score = f1_score(y_rf_binary, pred)
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"\nBest OOF threshold by F1: {best_threshold:.3f}")
print(f"Best OOF F1: {best_f1:.4f}")
print(f"OOF ROC-AUC: {roc_auc_score(y_rf_binary, invalid_probability_oof):.4f}")

## OOF Invalid Probability

| Metric | Value |
|---|---:|
| Minimum probability | 0.0000 |
| Maximum probability | 1.0000 |
| Mean probability | 0.1367 |
| Best F1 threshold | 0.415 |
| Best OOF F1 | 0.9695 |
| OOF ROC-AUC | 0.9935 |

In [ ]:
# ============================================================
# SANITY CHECKS
# ============================================================

print("SANITY CHECKS")
print("=============")
print("\n1. RF class order:")
print(best_rf.classes_)

print("\n2. OOF probability range:")
print(f"Min    : {invalid_probability_oof.min():.4f}")
print(f"Max    : {invalid_probability_oof.max():.4f}")
print(f"Mean   : {invalid_probability_oof.mean():.4f}")

print("\n3. OOF classification summary:")
print(pd.Series((invalid_probability_oof >= best_threshold).astype(int)).map({0: "Valid", 1: "Invalid"}).value_counts())

print("\n4. Actual target distribution:")
print(pd.Series(y_rf_binary).map({0: "Valid", 1: "Invalid"}).value_counts())

## Sanity Checks

1. **Random Forest class order**

    `[0, 1]`, where `0 = Valid` and `1 = Invalid`.

2. **OOF invalid-probability range**

    - Minimum: `0.0000`
    - Maximum: `1.0000`
    - Mean: `0.1367`

3. **OOF classification summary**

    | Prediction | Count |
    |---|---:|
    | Valid | 872 |
    | Invalid | 128 |

4. **Actual target distribution**

    | Class | Count |
    |---|---:|
    | Valid | 866 |
    | Invalid | 134 |

The predicted and actual class distributions are similar, indicating no obvious class-distribution anomaly in the out-of-fold predictions.